# Where Does RLHF Fit in LLM Training?

## Pretraining teaches language, SFT teaches task-following, human preference data teaches what people prefer, and RLHF uses that signal to refine behavior.

#  Part 4 — The Complete LLM Training Pipeline

So far, we studied RLHF almost in isolation.
        But an LLM does NOT usually begin with RLHF.

A typical training pipeline looks more like this:

                                                            Raw Text
                                                               ↓
                                                            Pretraining
                                                               ↓
                                                            Base LLM
                                                               ↓
                                                            Supervised Fine-Tuning
                                                               ↓
                                                            Instruction-Following Model
                                                               ↓
                                                            Human Preferences
                                                               ↓
                                                            Reward Model
                                                               ↓
                                                            Reinforcement Learning
                                                               ↓
                                                            Post-Trained Assistant

In [ ]:
# Stage 1: Pretraining

# Stage 1 — Pretraining

Before the model can follow instructions, it must first learn
patterns in language.

The core task is surprisingly simple:

> **Predict the next token.**

Example:

"The capital of France is ____"

The model tries to predict:
                Paris

### Pretraining is where the model learns massive amounts of linguistic and world-pattern knowledge from large text corpora.

## Tiny Next-Token Example 

In [2]:
sentence = "The cat sat on the"

possible_next_tokens = {
    "mat": 0.60,
    "table": 0.20,
    "moon": 0.01,
    "floor": 0.19
}

print("Context:")
print(sentence)

print("\nNext-token probabilities:\n")

for token, probability in possible_next_tokens.items():
    print(f"{token:10} → {probability:.0%}")

Context:
The cat sat on the

Next-token probabilities:

mat        → 60%
table      → 20%
moon       → 1%
floor      → 19%


## Pretraining Objective

#  Pretraining Objective

During pretraining, the model repeatedly sees text like:
    
    "Machine learning is a branch of ______"

The correct next token might be:
    "artificial"

    The model predicts probabilities.

    Then the model is updated so that the correct token becomes more likely.

    This happens across enormous amounts of text.

The result is:

# Base LLM

## A base LLM is good at continuing text, but that does not automatically mean it is good at behaving like a helpful assistant.

#  Imagine a Base Model

Prompt:

"Explain photosynthesis to a child."

A base model might continue with:

"Explain photosynthesis to a child.
Photosynthesis is commonly defined as..."

Or it might produce:

"Here are several exam questions about photosynthesis..."

Or continue in some other statistically likely direction.

Why?

Because its original training objective was:

> **Predict the next token**

not:

> **Follow the user's instruction perfectly**

# Stage 2 — Supervised Fine-Tuning (SFT)

Now we want the model to behave more like an assistant.

So we provide examples such as:

### Instruction

"Explain gravity to a child."

### Desired Response

"Gravity is like an invisible pull that keeps us on Earth..."

The model learns from many:

Instruction → Good Response

examples.

In [3]:
# SFT Dataset Example

sft_examples = [
    {
        "instruction": "Explain gravity simply.",
        "response": "Gravity is the force that pulls objects toward each other."
    },
    {
        "instruction": "Summarize this paragraph.",
        "response": "The paragraph describes how plants convert sunlight into energy."
    },
    {
        "instruction": "Write a polite email declining a meeting.",
        "response": "Thank you for the invitation. Unfortunately, I won't be able to attend..."
    }
]

for example in sft_examples:
    print("\nINSTRUCTION:")
    print(example["instruction"])

    print("\nGOOD RESPONSE:")
    print(example["response"])

    print("-" * 50)


INSTRUCTION:
Explain gravity simply.

GOOD RESPONSE:
Gravity is the force that pulls objects toward each other.
--------------------------------------------------

INSTRUCTION:
Summarize this paragraph.

GOOD RESPONSE:
The paragraph describes how plants convert sunlight into energy.
--------------------------------------------------

INSTRUCTION:
Write a polite email declining a meeting.

GOOD RESPONSE:
Thank you for the invitation. Unfortunately, I won't be able to attend...
--------------------------------------------------


In [ ]:
## In SFT, the desired answer is explicitly shown to the model.
That's different from RLHF, where humans can simply express preferences among alternatives.

#   What Does SFT Teach?

Pretraining teaches:

> "What text is likely?"

SFT teaches:

> "When a user asks this kind of question,
> respond in this kind of way."

So the model becomes better at:

- Following instructions
- Answering questions
- Formatting responses
- Conversational behavior
- Performing requested tasks

#  Pretraining vs SFT

| Stage | Training Signal | Main Goal |
|---|---|---|
| Pretraining | Next-token prediction | Learn language and patterns |
| SFT | Demonstration response | Learn instruction-following behavior |

But there is still a problem...

A response can be:

- Correct but unnecessarily verbose
- Helpful but badly formatted
- Safe but unhelpful
- Technically accurate but confusing
- Fluent but not what users prefer

How do we teach the model these preferences?

# Stage 3 — Generate Multiple Responses

Suppose our SFT model receives:

> "Explain neural networks to a beginner."

It generates several possible responses:

Response A

Response B

Response C

Response D

Instead of humans writing another perfect response from scratch...

we can ask them:

> **Which response do you prefer?**

In [4]:
prompt = "Explain neural networks to a beginner."

responses = {
    "A": "Neural networks are universal function approximators composed of parameterized nonlinear transformations.",
    
    "B": "A neural network is a computer system that learns patterns from examples, a bit like learning to recognize cats after seeing many cat pictures.",
    
    "C": "Neural networks are algorithms.",
    
    "D": "Deep learning consists of interconnected computational units with weighted activation functions."
}

print("PROMPT:")
print(prompt)

for label, response in responses.items():
    print(f"\nResponse {label}:")
    print(response)

PROMPT:
Explain neural networks to a beginner.

Response A:
Neural networks are universal function approximators composed of parameterized nonlinear transformations.

Response B:
A neural network is a computer system that learns patterns from examples, a bit like learning to recognize cats after seeing many cat pictures.

Response C:
Neural networks are algorithms.

Response D:
Deep learning consists of interconnected computational units with weighted activation functions.


In [ ]:
preference = input(
    "\nWhich response do you prefer? A/B/C/D: "
).upper()

print("\nHuman preferred response:", preference)

#   Human Preference Dataset

The data might look like:

Prompt 1:
B > A

Prompt 2:
C > D

Prompt 3:
A > B

Prompt 4:
D > C

...

This dataset captures judgments about qualities such as:

Helpful

Clear

Relevant

Accurate

Safe

Appropriately detailed

## The human preference dataset is not yet the reinforcement-learning signal. We first need to turn preferences into something the RL algorithm can use.

#  Stage 4 — Train the Reward Model

The preference data is used to train "another" model:
        
        Prompt + Response
                ↓
        Reward Model
                ↓
        Numerical Score


Example:

        Response A → 0.21
        
        Response B → 0.87

If humans consistently prefer B...

the Reward Model learns to assign B a higher score.

## Is this reward model same LLM ?

#  Two Different Models

## Policy Model

The LLM that generates responses.

                Prompt
                
                   ↓
                   
                Policy Model
                
                   ↓
                   
                Response


## Reward Model

The model that evaluates responses.

                Prompt + Response
                
                      ↓
                      
                Reward Model
                
                      ↓
                      
                    Reward

# Stage 5 — Reinforcement Learning

Now the loop begins.

                Prompt
                   ↓
                Policy LLM
                   ↓
                Response
                   ↓
                Reward Model
                   ↓
                Reward
                   ↓
                RL Algorithm
                   ↓
                Update Policy LLM


# Don't Forget the Reference Model

Before RL training starts, we usually keep a frozen copy of
the model.

This becomes the:

# Reference Model

The policy is allowed to improve...

but we penalize it for moving too far away from the
reference behavior.

So the objective is roughly:

Reward
   minus
KL Penalty

# Classic RLHF Pipeline

In [ ]:
                    ┌──────────────────────┐
                    │   LARGE TEXT DATA    │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │     PRETRAINING      │
                    │ Next-token prediction│
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │       BASE LLM       │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │         SFT          │
                    │ Instruction → Answer │
                    └──────────┬───────────┘
                               │
                               ▼
                    ┌──────────────────────┐
                    │      SFT MODEL       │
                    └───────┬──────────────┘
                            │
               ┌────────────┴────────────┐
               │                         │
               ▼                         ▼
       Candidate Responses        Reference Model
               │
               ▼
        Human Comparisons
               │
               ▼
       Preference Dataset
               │
               ▼
        ┌───────────────┐
        │ Reward Model  │
        └───────┬───────┘
                │
              Reward
                │
                ▼
        ┌───────────────┐
        │ RL / PPO      │◄──── KL penalty
        └───────┬───────┘
                │
                ▼
        Update Policy LLM
                │
                ▼
        Post-Trained Model

#  Where Does RLHF Begin?

Look carefully:

                    Pretraining
                       ↓
                    SFT
                       ↓
                    Human Preference Collection
                       ↓
                    Reward Model
                       ↓
                    RL
                       ↓
                    Final Model

The term **RLHF** usually refers to the post-training process where human preference information is used to provide the
training signal for reinforcement learning.

So:

## Pretraining ≠ RLHF

## SFT ≠ RLHF

They usually happen before it.

#  TO CHECK WHETHER YOU HAD UNDERSTOOD 

For each stage, decide whether it is:

PRETRAINING

SFT

HUMAN FEEDBACK

REWARD MODEL TRAINING

RL


### Scenario 1

Model predicts the next word in billions of documents.

### Scenario 2

Humans write examples of high-quality assistant responses.

### Scenario 3

Humans choose Response B over Response A.

### Scenario 4

A model learns to assign higher scores to preferred responses.

### Scenario 5

The language model's parameters are updated using rewards.

In [ ]:
# Trace One Prompt Through the Pipeline

Prompt:

> "Explain black holes to a 12-year-old."

In [ ]:
# STEP 1
Base Model

Has learned language patterns from pretraining.

# STEP 2
SFT Model

Has learned how to respond to instructions.

# STEP 3
Generate several candidate responses.

A
B
C
D

# STEP 4
Humans compare responses.

B > A

B > C

D > A

# STEP 5
Reward Model learns the preference patterns.
    
# STEP 6
Policy generates a new response.

Reward Model scores it.
    
# STEP 7
RL updates the policy.

Responses predicted to be more preferred
become more likely over training.

In [ ]:
#  Common Confusions

## Pretraining

Does NOT require humans ranking assistant responses.
It mainly learns from large-scale text prediction.


## SFT

Does NOT require a reward model.
It learns directly from demonstrations.


## Reward Model

Does NOT usually generate the final assistant response.
It evaluates responses.


## RL

Does NOT create the human preference dataset.
It uses a reward signal derived from preference information.


## RLHF

Is NOT the entire process of training an LLM from scratch.

In [ ]:
# Why Not Just Keep Doing SFT?

Suppose humans prefer:

Response B > Response A

With supervised fine-tuning, we might need to provide a specific desired answer.

With preference learning, humans can simply say:

> "I prefer B."

This can capture preferences that are difficult to specify as exact target outputs.

For example:

- Which answer is clearer?
- Which one is less verbose?
- Which tone is better?
- Which answer follows the user's intent better?

#  The Key Insight

The stages teach different things.

## PRETRAINING

    "What does language look like?"


## SUPERVISED FINE-TUNING

    "How should I respond to instructions?"


## HUMAN FEEDBACK

    "Which responses do people prefer?"


## REWARD MODEL

    "Can I predict those preferences?"


## REINFORCEMENT LEARNING

    "Can I change the policy so preferred responses become more likely?"

#  A Compact View

Let:

πθ(y | x)

be the probability that the LLM generates response **y**
for prompt **x**.

Let:

r(x, y)

be the reward predicted for that response.

RLHF roughly tries to improve:

    Expected Reward

while preventing the policy from drifting too far from the reference model.

Conceptually:

maximize

            E[r(x, y)] - β KL(πθ || πref)

#   Important: This Is the Classic RLHF Pipeline

The pipeline we have studied is commonly used to explain
classic RLHF:

Human Preferences

      ↓
Reward Model

      ↓
Reinforcement Learning

      ↓
Policy Update

However, modern LLM post-training can use other methods too.

Not every preference-trained model necessarily uses
the exact:

        Reward Model + PPO

pipeline.

We will examine one important alternative next:

# DPO

In [ ]:
#question 

Put these stages in the correct order:

A. Reward Model Training

B. Pretraining

C. Reinforcement Learning

D. Human Preference Collection

E. Supervised Fine-Tuning

F. Base Language Model

#  RLHF So Far

## Part 1 — Reinforcement Learning

Action → Reward → Change Policy


## Part 2 — Human Feedback

Humans compare responses

→ Build preference data

→ Train a Reward Model


## Part 3 — Policy Optimization

Reward

→ Advantage / RL algorithm

→ Update LLM policy


## Part 4 — LLM Training Pipeline

Pretraining

→ SFT

→ Human Preferences

→ Reward Model

→ RL

→ Post-Trained Assistant

#  But Could We Skip Some of This?

We currently have:

Human Preferences

      ↓
Reward Model

      ↓
PPO

      ↓
Updated LLM

But...

What if we could train the LLM **directly from the
preference pairs**?

Human says:

B > A

Could the LLM learn directly from that?

Without first training a separate Reward Model?

Without running PPO?

## Yes.

This leads us to:

# Direct Preference Optimization — DPO